In [ ]:
from icecream import ic
from tqdm import tqdm
import httpx

In [ ]:
tqdm.pandas(desc="Processing files")

In [ ]:
import threading

def minha_funcao():
    """Display a message from a thread."""
    display("Esta é uma thread")


# Criação de uma instância de Thread
thread = threading.Thread(target=minha_funcao)


# Inicia a thread
thread.start()

# Espera pela thread terminar
thread.join()

display("Thread principal finalizada")

## Comunicação Entre Threads


import threading
import queue
import time

def produtor(fila):
    for i in range(5):
        time.sleep(1)  # Simula algum trabalho
        mensagem = f"Mensagem {i}"
        fila.put(mensagem)
        ic(f"Produzido: {mensagem}")


def consumidor(fila):
    while True:
        mensagem = fila.get()
        ic(f"Consumido: {mensagem}")
        fila.task_done()

        
fila = queue.Queue()
produtor_thread = threading.Thread(target=produtor, args=(fila,))
consumidor_thread = threading.Thread(target=consumidor, args=(fila,))
produtor_thread.start()
consumidor_thread.start()
produtor_thread.join()
consumidor_thread.join()
ic("Threads finalizadas")

In [ ]:
import multiprocessing 

def spawn(num): 
    print(num) 
    

if __name__ == '__main__': 
    for i in range(5): 
        p = multiprocessing.Process(target=spawn, args=(i,)) 
        p.start() 
        p.join() # this line allows you to wait for processes


In [ ]:
import threading


def print_cube(num):
    print("Cube: {}" .format(num * num * num))


def print_square(num):
    print("Square: {}" .format(num * num))


if __name__ =="__main__":
    t1 = threading.Thread(target=print_square, args=(10,))
    t2 = threading.Thread(target=print_cube, args=(10,))

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    print("Done!")

In [ ]:
import threading
import os

def task1():
    print("Task 1 assigned to thread: {}".format(threading.current_thread().name))
    print("ID of process running task 1: {}".format(os.getpid()))

def task2():
    print("Task 2 assigned to thread: {}".format(threading.current_thread().name))
    print("ID of process running task 2: {}".format(os.getpid()))

if __name__ == "__main__":

    print("ID of process running main program: {}".format(os.getpid()))

    print("Main thread name: {}".format(threading.current_thread().name))

    t1 = threading.Thread(target=task1, name='t1')
    t2 = threading.Thread(target=task2, name='t2')

    t1.start()
    t2.start()

    t1.join()
    t2.join()

In [ ]:
import concurrent.futures

def worker():
    print("Worker thread running")

pool = concurrent.futures.ThreadPoolExecutor(max_workers=2)

pool.submit(worker)
pool.submit(worker)

pool.shutdown(wait=True)

print("Main thread continuing to run")

In [ ]:
from pathlib import Path


In [ ]:
csv_file = Path.cwd().resolve().parents[4]/'data_files'/'csv'/'atos20250801.csv'
csv_file.is_file()

In [ ]:
import pandas as pd
df0 = pd.read_csv(csv_file, sep=';', encoding='utf-8', low_memory=False)
df0.tail()

In [ ]:
from urllib.parse import urljoin

urljoin('https://www.planalto.gov.br/', 'CCIVIL_03')

In [ ]:
df0['url_ato'] = df0.URI.apply(lambda x: urljoin('https://www.planalto.gov.br/', x))

In [ ]:
df0

In [ ]:
from pathlib import Path 
import time
from concurrent.futures import ThreadPoolExecutor
import pandas as pd


files = Path.cwd().resolve().parents[4].joinpath('data_files','xlsx').glob('*.xlsx')

def fetch_data(file):
    print(f"{file} started")
    wb = pd.ExcelFile(file)
    sheet = [x for x in wb.sheet_names if 'sum' in x.lower()][0]
    df = wb.parse(sheet)
    wb.close()
    print(df.shape)
    return df.shape

start = time.perf_counter()
with ThreadPoolExecutor() as executor:
    dfs = [executor.submit(fetch_data, file) for file in files]
end = time.perf_counter()
print(f'Finished in {end-start} seconds')

In [ ]:
import concurrent.futures
import pandas as pd
import time
from pathlib import Path

def read_file(file):
    print(f"{file} started")
    wb = pd.ExcelFile(file)
    sheet = [x for x in wb.sheet_names if 'sum' in x.lower()][0]
    df = wb.parse(sheet)
    wb.close()
    print(df.shape)
    return df.shape

if __name__ == '__main__':
    files = Path.cwd().resolve().parents[4].joinpath('data_files','xlsx').glob('*.xlsx')
    start = time.perf_counter()
    with concurrent.futures.ProcessPoolExecutor() as executor:
        dfs = [executor.submit(read_file, file) for file in files]
    end = time.perf_counter()
    print(f'Finished in {end-start} seconds')

In [ ]:

def accessible_url(url: str, timeout: int = 1, headers: dict|None = None) -> bool:
    """Check if a URL is accessible within a given timeout."""
    headers = headers or {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36'}
    timeout = timeout or 1
    ic(f'Starting {url}')
    try:
        response = httpx.get(url, timeout=timeout, headers=headers)
    except httpx.RequestError:
        return False
    else:
        ic(f'.. {url} finished.')
        return response.status_code == httpx.codes.OK

In [ ]:
accessible_url('https://www.planalto.gov.br/CCIVIL_03/')

In [ ]:
r= httpx.get('https://www.planalto.gov.br/CCIVIL_03/', timeout=10, headers={'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36'})
r

__5 por segundo__

```python
df0['readable'] = df0.url_ato.progress_apply(lambda x: accessible_url(x))
```

In [ ]:
df0.url_ato.to_list()

Progress bar into multiprocess fail.

with concurrent.futures.ProcessPoolExecutor() as executor:
    result = [executor.submit(accessible_url, url) for url in tqdm(df0.url_ato.to_list())]

with concurrent.futures.ProcessPoolExecutor() as executor:
    result = list(tqdm(executor.submit(accessible_url, url) for url in df0.url_ato.to_list()))